# nb01 · Tái lập Bảng 1 (VGG16) của *Hidden Monotonicity*

**Mục tiêu.** Chạy lại code chính thức của Zimmermann & Loho (CVPR 2026), repo `DC-Decomposition-for-Explanability` ở commit `31be19b`, để tái lập các hàng sau của Bảng 1 trên 566 ảnh test của ImageNet-S-50, với đủ 6 thước đo Quantus:
- 6 hàng Split;
- 2 hàng LayerCAM (tầng 14 và tầng 2);
- hàng Random.

Kết quả là baseline cho mọi so sánh về sau.

**Nội dung.**
1. Kiểm tra môi trường, phiên bản repo HM và dữ liệu (616 ảnh = 50 val + 566 test).
2. Sinh bản đồ giải thích bằng chính code HM. Notebook không sửa dòng nào của repo, chỉ truyền đường dẫn, dtype và danh sách ảnh.
3. Tính 4 thước đo (Selectivity, Attr. Loc., Pointing Game, Pixel Flipping), rồi Max Sensitivity.
4. So với Bảng 1 và tự động kết luận **THÀNH CÔNG / MỘT PHẦN / THẤT BẠI**.

**Cách chạy.** Chỉ đổi `N_IMAGES` ở §1: `1` để chạy thử, `566` để chạy đủ. Kết quả lưu ở `results/nb01/n{N_IMAGES}/`. Khi chạy lại, những bản đồ và thước đo đã có được lấy từ cache, nên một lần chạy bị ngắt có thể chạy tiếp.

**Chạy song song trên một GPU.** Ba bước nặng (§6 sinh bản đồ, §7 bốn thước đo, §8 Max Sens) tự mở `N_WORKERS` tiến trình con (`scripts/nb01_worker.py`) trên GPU số `GPU`.
- Worker w xử lý các ảnh k có k mod `N_WORKERS` = w.
- Notebook hiện một thanh tiến trình tổng, cộng dồn từ mọi worker.
- Nếu một worker lỗi, notebook dừng và in phần cuối log của worker đó.
- Ảnh k luôn dùng seed 42·10⁴ + k, nên kết quả không phụ thuộc số worker.

**Chạy từ terminal, có thanh tiến trình** (từ thư mục `gauge-centering-xai/experiments`):
```
uv run python scripts/run_notebook.py nb01                                         # cấu hình như trong notebook
uv run python scripts/run_notebook.py nb01 --n-images 566 --gpu 1 --workers 3      # đổi cấu hình cho lần chạy này
```
Bản notebook đã chạy được lưu ở `results/nb01/n{N}/nb01_hm_table1_vgg16.executed.ipynb`; log của các worker ở `results/nb01/n{N}/workers/`.

**Lưu kết quả, tách tính toán và hiển thị.** Mỗi section gồm hai loại cell:
- **Cell tính toán:** tính rồi lưu số liệu gốc vào `data/` (CSV/NPZ, tên cột cố định như `row`, `metric`, `passed`).
- **Cell hiển thị:** dòng đầu là `# ▶ Hiển thị`; cell này chỉ đọc file trong `data/` rồi hiện bảng hoặc vẽ hình qua `scripts/nb01_report.py`.

Tên cột, tên hàng và tiêu đề hình được gán lúc hiển thị, qua `LABELS` (§14). Muốn đổi thì sửa `LABELS` rồi chạy lại các cell hiển thị, không phải tính lại.

**Quy tắc tái lập:**
- **Mức 1, từng ô:** ô đạt nếu $|\text{ta} - \text{HM}| \le 2\,\mathrm{SE}$, với $\mathrm{SE} = s/\sqrt n$ và $s$ là độ lệch chuẩn của điểm theo từng ảnh. Ô có $z = |\Delta|/\mathrm{SE} > 0.5$ vẫn đạt nhưng được gắn cờ "cần ghi chú".
  - Max Sens chỉ chạy seed 42, giống HM. SE theo ảnh đã bao gồm nhiễu Monte Carlo của từng ảnh. Hơn nữa, ta và HM dùng cùng 566 ảnh, nên phần chênh do nhiễu nhỏ hơn SE rất nhiều.
- **Mức 2, kết luận của bảng:**
  - (a) hàng dẫn đầu ở mỗi thước đo không đổi; tính là hoà nếu khoảng cách nằm trong 2·SE của hiệu ghép cặp theo ảnh;
  - (b) dấu của hiệu (Split tốt nhất − LayerCAM tốt nhất) không bị đảo một cách có ý nghĩa;
  - (c) Random kém nhất ở hai thước đo định vị.
- **Kết luận:** THÀNH CÔNG nếu mức 1 và mức 2 đều đạt. MỘT PHẦN nếu mức 2 đạt nhưng có ô trượt mức 1. THẤT BẠI nếu có ít nhất một kết luận mức 2 bị đảo.

## 1. Cấu hình

Chỉ cần đổi `N_IMAGES`. Mọi tham số khác đều có lý do ghi ở bảng bên dưới. Cell này không tính gì, chỉ đặt đường dẫn, nên chạy lại bất cứ lúc nào cũng được.

**float32 cho mọi phương pháp.** Bài viết ghi "All intermediate computations use torch.float64", nhưng ta chạy float32 vì:
- trên GPU T4, float64 chậm hơn float32 khoảng 30 lần; ở float64, chạy đủ mất khoảng 920 giờ;
- CLI của HM mặc định là float32, và chính Bảng 7 của HM được đo ở float32 (`timing_evaluation.py`);
- ở float64, code HM không tính được Max Sens của LayerCAM (lỗi kiểu `float`/`double`, HM ghi NaN), vậy các hàng baseline của Bảng 1 chắc chắn đã chạy ở float32.

Đây là một điểm lệch so với bài và phải được ghi khi báo cáo.

**Chế độ chạy thử** (`N_IMAGES < 566`): Max Sensitivity dùng 2 mẫu nhiễu thay vì 25, nên kết luận của lần chạy thử không có giá trị; nó chỉ chứng minh mọi cell chạy được. Với 1 ảnh, SE không tính được nên §11 ghi "không chấm được".

**GPU và số worker.**
- `GPU` chọn GPU cho cả kernel lẫn worker (đặt `CUDA_VISIBLE_DEVICES` trước khi import torch).
- `N_WORKERS` là số tiến trình chạy song song trên GPU đó.

Mỗi worker cần bộ nhớ GPU cho model VGG16 cùng cặp luồng tách. Đỉnh bộ nhớ thật của từng worker được đo và hiện ở §6, để chỉnh `N_WORKERS` cho vừa phần bộ nhớ còn trống. Hai giá trị này đọc được từ biến môi trường `NB_GPU`, `NB_WORKERS`; `scripts/run_notebook.py --gpu --workers` đặt các biến đó.

In [ ]:
import os, sys, json, time, shutil, subprocess, platform, logging, warnings, contextlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

N_IMAGES = 1          # <- số duy nhất cần đổi: 1 = chạy thử, 566 = toàn bộ tập test

N_TEST = 566
assert 1 <= N_IMAGES <= N_TEST
SMOKE = N_IMAGES < N_TEST
GROUPS = {            # thư mục con: (dtype, các file config phương pháp)
    "fp32": ("float32", ["splitcam_a40_sc.json", "splitcam_a40_sc_wta_f26.json", "splitgrad_a40_wta.json",
                         "splitlrp_sc_wta.json", "layercam.json", "random_baseline.json"]),
}
SPLIT_SEED = 42
INNERLOOP_N = 1
MAXSENS_NR_SAMPLES = 2 if SMOKE else 25
MAXSENS_SEEDS = [42]
HM_COMMIT = "31be19b460358a69516452402d41bd650845a344"

GPU = int(os.environ.get("NB_GPU", 1))           # GPU dùng cho kernel và các worker
N_WORKERS = int(os.environ.get("NB_WORKERS", 3))  # số tiến trình song song trên GPU đó
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)     # phải đặt trước khi import torch (§2)

PARAMS = pd.DataFrame([
    ("N_IMAGES", N_IMAGES, "1 = chạy thử; 566 = toàn bộ tập test của bài (HM §5)"),
    ("dtype", "float32", "lệch khỏi bài (bài ghi float64): float64 trên T4 chậm hơn ~30 lần (~920 h); CLI của HM mặc định "
                         "float32, Bảng 7 của HM đo ở float32; ở float64 HM trả NaN cho Max Sens của LayerCAM"),
    ("SPLIT_SEED", SPLIT_SEED, "seed mặc định của HM (--random-seed); quyết định 50 ảnh val, từ đó 566 ảnh test; "
                               "ảnh k dùng seed 42·10⁴ + k cho Random và nhiễu Max Sens, nên kết quả không phụ thuộc cách chia phần"),
    ("INNERLOOP_N", INNERLOOP_N, "HM dọn bộ nhớ GPU sau mỗi INNERLOOP_N ảnh (mặc định 50); với 50, GPU 4 GB hết bộ nhớ "
                                 "ở ảnh thứ 2. 1 = dọn sau mỗi ảnh; thứ tự xử lý và kết quả không đổi"),
    ("MAXSENS_NR_SAMPLES", MAXSENS_NR_SAMPLES, "HM §5: 25 mẫu nhiễu mỗi ảnh (chạy thử: 2)"),
    ("MAXSENS_SEEDS", MAXSENS_SEEDS, "như HM: một lần chạy seed 42; SE theo ảnh đã gồm nhiễu Monte Carlo"),
    ("GPU", GPU, "server: GPU 0 đang bận (100%), GPU 1 rảnh nhưng chỉ còn ~13.8 GB trống (người khác giữ 35 GB)"),
    ("N_WORKERS", N_WORKERS, "~13.8 GB trống chia cho ~4 GB/worker ở float32 (ước từ 8.3 GB ở float64); Quantus "
                             "chủ yếu dùng CPU nên nhiều worker giúp GPU bận hơn; xem đỉnh bộ nhớ đo được ở §6 để chỉnh"),
    ("HM_COMMIT", HM_COMMIT[:7], "phiên bản repo HM mà notebook này được viết theo"),
], columns=["param", "value", "reason"])

NB_ID = "nb01"         # mã notebook: configs/, reference/, results/ và tên hình, bảng đều mang mã này
EXP_DIR = Path.cwd().resolve()
assert (EXP_DIR / "pyproject.toml").exists(), "hãy mở notebook từ thư mục gauge-centering-xai/experiments"
DOCS = EXP_DIR.parents[1]
HM_REPO = Path(os.environ.get("HM_REPO", DOCS / "DC-Decomposition-for-Explanability")).resolve()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", DOCS / "data")).resolve()
IMAGENET, IMAGENET_S = DATA_ROOT / "imagenet", DATA_ROOT / "imagenet-s"
CFG_DIR = EXP_DIR / "configs" / NB_ID
REF_CSV = EXP_DIR / "reference" / NB_ID / "hm_table1_vgg16.csv"
RUN_DIR = EXP_DIR / "results" / NB_ID / f"n{N_IMAGES}"
MAIN_DIR = RUN_DIR / f"seed{SPLIT_SEED}"          # bản đồ + 6 thước đo (seed 42), mỗi nhóm dtype một thư mục con
DATA_DIR = RUN_DIR / "data"                       # số liệu gốc của mọi bảng và hình
for d in (DATA_DIR, RUN_DIR / "figures", RUN_DIR / "tables"):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(EXP_DIR / "scripts"))
import nb01_report as rep                         # bảng và hình, chỉ đọc từ DATA_DIR
from nb_progress import Progress                  # thanh tiến trình + progress.json cho scripts/run_notebook.py
LABELS = rep.default_labels()                     # tên hiển thị; sửa ở §14

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 100)
PARAMS.to_csv(DATA_DIR / "params.csv", index=False)
print(f"N_IMAGES = {N_IMAGES} ({'CHẠY THỬ' if SMOKE else 'CHẠY ĐỦ'}); GPU {GPU}, {N_WORKERS} worker")
print(f"repo HM : {HM_REPO}\ndữ liệu : {DATA_ROOT}\nkết quả : {RUN_DIR}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "params.csv", LABELS))

## 2. Môi trường và repo HM

Kiểm tra repo HM đúng commit `31be19b` và không có thay đổi cục bộ, kiểm tra có GPU CUDA. Phiên bản các thư viện được ghi vào `data/env.json`.

Code HM gọi `logging.basicConfig(level=DEBUG)`, nên nếu để nguyên, mọi dòng log sẽ in ra notebook. Trên console ta chỉ giữ log mức ERROR (ví dụ CUDA out of memory); log đầy đủ vẫn nằm trong `evaluation.log` của từng thư mục chạy. Phần in ra màn hình của HM (print, tqdm) được chuyển vào `console.log`.

In [ ]:
warnings.filterwarnings("ignore", category=SyntaxWarning)   # Quantus 0.6 docstrings
warnings.filterwarnings("ignore", message="IProgress not found")
sys.path.insert(0, str(HM_REPO))
import torch, torchvision, quantus, captum, zennit, torch_sparse, torch_scatter
from xai.evaluation.dataset_utils import _load_dataset
from xai.evaluation.core import ResultsManager
for h in logging.getLogger().handlers:
    h.setLevel(logging.ERROR)                               # HM errors (e.g. CUDA OOM) still show
LOG = logging.getLogger("hm_repro")

# HM opens its log files with the locale encoding; on Windows (cp1252) its emoji log lines fail and
# flood stderr with "Logging error" tracebacks. Default every FileHandler to UTF-8 (no effect on Linux).
_file_handler_init = logging.FileHandler.__init__
def _utf8_file_handler_init(self, filename, mode="a", encoding=None, *args, **kwargs):
    _file_handler_init(self, filename, mode, encoding or "utf-8", *args, **kwargs)
logging.FileHandler.__init__ = _utf8_file_handler_init

def git(*args):
    return subprocess.run(["git", "-C", str(HM_REPO), *args], capture_output=True, text=True, check=True).stdout.strip()

commit, modified = git("rev-parse", "HEAD"), git("status", "--porcelain", "--untracked-files=no")
assert commit == HM_COMMIT, f"repo HM đang ở {commit}, cần {HM_COMMIT}"
assert modified == "", f"repo HM có thay đổi cục bộ:\n{modified}"
assert torch.cuda.is_available(), "cần GPU CUDA"

ENV = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "torchvision": torchvision.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
    "numpy": np.__version__, "pandas": pd.__version__, "quantus": quantus.__version__,
    "captum": captum.__version__, "zennit": getattr(zennit, "__version__", "?"),
    "torch_sparse": torch_sparse.__version__, "hm_commit": commit, "n_images": N_IMAGES, "smoke": SMOKE,
    "gpu_index": GPU, "gpu_free_gib_at_start": round(torch.cuda.mem_get_info()[0] / 2**30, 1), "n_workers": N_WORKERS,
}
(DATA_DIR / "env.json").write_text(json.dumps(ENV, indent=2))

@contextlib.contextmanager
def to_console_log():
    # HM prints progress with print()/tqdm; keep it out of the notebook
    with open(RUN_DIR / "console.log", "a", encoding="utf-8") as fh, \
         contextlib.redirect_stdout(fh), contextlib.redirect_stderr(fh):
        yield
print(f"môi trường và repo HM: đạt; {ENV['gpu']} (GPU {GPU}), còn trống {ENV['gpu_free_gib_at_start']} GiB")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_env(RUN_DIR, LABELS))

## 3. Dữ liệu: 616 ảnh = 50 val + 566 test

Loader của HM đọc từng mask trong `validation-segmentation/<wnid>/` và tìm ảnh ở `val/<tên ảnh>` hoặc `val/<wnid của mask>/<tên ảnh>`. Ảnh ImageNet được xếp theo thư mục nhãn gốc (`scripts/prepare_data.py`). Vì vậy 136 mask mà ImageNet-S xếp vào lớp khác với nhãn ImageNet sẽ không tìm thấy ảnh và bị bỏ qua, còn lại đúng 616 ảnh như bài. Sau đó HM chọn 50 ảnh val (mỗi lớp 1 ảnh, seed 42); 566 ảnh còn lại là tập test.

Cell tính toán kiểm tra các con số trên và lưu danh sách ảnh vào `data/image_split.csv`. Lần chạy thử dùng `N_IMAGES` ảnh **đầu tiên** trong danh sách 566 ảnh test, nên chúng là tập con của lần chạy đủ.

In [ ]:
masks = sorted((IMAGENET_S / "ImageNetS50" / "validation-segmentation").glob("*/*.png"))
with to_console_log():
    dataset, test_idx = _load_dataset(True, str(IMAGENET), str(IMAGENET_S), "ImageNetS50", -1, SPLIT_SEED, False, LOG)
    val_idx = dataset.get_stratified_indices(n_samples=50, random_seed=SPLIT_SEED)
test_idx = [int(i) for i in test_idx]
cls = lambda idx: [dataset.samples[i]["class_id"] for i in idx]

checks = pd.Series({
    "752 mask ImageNet-S-50": len(masks) == 752,
    "616 ảnh nạp được (mask khớp nhãn ImageNet)": len(dataset) == 616,
    "val: 50 ảnh, mỗi lớp 1 ảnh": len(val_idx) == 50 and len(set(cls(val_idx))) == 50,
    "test: 566 ảnh": len(test_idx) == N_TEST,
    "val và test rời nhau, hợp lại đủ 616": set(val_idx).isdisjoint(test_idx) and len(set(val_idx) | set(test_idx)) == 616,
})
assert checks.all(), checks[~checks]

checks.rename_axis("check").rename("passed").reset_index().to_csv(DATA_DIR / "data_checks.csv", index=False)
split = pd.DataFrame(
    [{"image_name": dataset.samples[i]["image_name"], "class_id": dataset.samples[i]["class_id"],
      "split": "val", "test_position": None} for i in val_idx] +
    [{"image_name": dataset.samples[i]["image_name"], "class_id": dataset.samples[i]["class_id"],
      "split": "test", "test_position": k} for k, i in enumerate(test_idx)])
split.to_csv(DATA_DIR / "image_split.csv", index=False)
# data of the example figure: first test image after HM's resize + crop, and its mask
img, mask, class_id, _, name = dataset[test_idx[0]]
img = torchvision.transforms.CenterCrop(224)(torchvision.transforms.Resize(256)(img))
np.savez_compressed(DATA_DIR / "fig_example_mask.npz", image=np.asarray(img), mask=np.asarray(mask, dtype=bool),
                    image_name=name, class_id=class_id)
print(f"dữ liệu: đạt; lần chạy này dùng {N_IMAGES} ảnh test đầu tiên")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "data_checks.csv", LABELS))
split_saved = pd.read_csv(DATA_DIR / "image_split.csv")
display(split_saved[split_saved.split == "test"].head(min(N_IMAGES, 10)))
rep.plot_example_mask(RUN_DIR, LABELS); plt.show()

## 4. Phương pháp: hàng Bảng 1 ↔ config

Config được chép từ repo HM vào `configs/nb01/methods/`, kèm ba điều chỉnh có ghi rõ:
- **Hàng 3**, SplitCAM (sc, α=0.4, wta) (26,+,g): repo không có config cho hàng này. Ta lấy tham số của config SplitCAM wta α=0.4 và thêm công thức `f26_cam_pp`/`abs`.
- **Hàng 6**, SplitLRP: bỏ công thức `f7` vì không có trong Bảng 1, để tiết kiệm thời gian.
- **Hàng 9**, Random: chép config của ResNet18 và đổi model thành `vgg16`.

Bản chụp của config được lưu vào `RUN_DIR/configs/methods/<nhóm>/`. Ngoài các hàng trên, HM còn tự lưu và chấm điểm hai "phương pháp" phụ là `baseline_original_image` và `baseline_binary_mask`; chúng không thuộc Bảng 1 và bị bỏ qua khi so sánh.

In [ ]:
REF = pd.read_csv(REF_CSV)
SNAP = RUN_DIR / "configs"
GROUP_OF = {f: g for g, (_, files) in GROUPS.items() for f in files}
assert set(GROUP_OF) == set(REF["config_file"]), "mỗi file config phải thuộc đúng một nhóm dtype"
REF["group"] = REF["config_file"].map(GROUP_OF)
for g, (_, files) in GROUPS.items():
    (SNAP / "methods" / g).mkdir(parents=True, exist_ok=True)
    for f in files:
        if not (SNAP / "methods" / g / f).exists():
            shutil.copy(CFG_DIR / "methods" / f, SNAP / "methods" / g / f)

rows = []
for r in REF.itertuples():
    cfg = json.loads((CFG_DIR / "methods" / r.config_file).read_text())
    formulas = {f["name"]: f for f in cfg.get("formulas", [])}
    found = cfg["name"] == r.method_name and (pd.isna(r.formula) or r.formula in formulas)
    op = formulas[r.formula]["op"] if isinstance(r.formula, str) and r.formula in formulas else "-"
    rows.append({"row": r.row, "table1": f"{r.label} {r.layer}", "abs_paper": r.abs, "group": r.group,
                 "config_file": r.config_file, "formula": r.formula if isinstance(r.formula, str) else "-",
                 "op": op, "found": found, "source": r.config_source})
METHOD_TABLE = pd.DataFrame(rows)
assert METHOD_TABLE["found"].all(), METHOD_TABLE[~METHOD_TABLE["found"]]
METHOD_TABLE.to_csv(DATA_DIR / "method_table.csv", index=False)
print(f"phương pháp: {len(METHOD_TABLE)} hàng, đủ config")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "method_table.csv", LABELS))

## 5. Thước đo

Năm file config thước đo được chép nguyên từ `quantus_evaluation_configs/metrics/enabled/` của repo HM, rồi chia làm hai nhóm:
- `metrics_4/`: Selectivity, Attr. Loc., Pointing Game, Pixel Flipping (§7);
- `metrics_maxsens/`: Max Sensitivity (§8).

Tách như vậy để đo riêng thời gian của Max Sens. Ở chế độ chạy thử, `nr_samples` của Max Sens được thay bằng `MAXSENS_NR_SAMPLES`.

Pixel Flipping trả về cả đường cong. HM tính AUC@5 và AUC@20 bằng hàm `compute_drop_metrics`: lấy tổng các độ giảm logit từ bước 0 đến bước ứng với 5% (hoặc 20%) số bước, rồi chia cho (số bước − 1). Mỗi bước thay 112 pixel bằng giá trị trung bình của ảnh.

In [ ]:
METRICS4_DIR, MAXSENS_DIR = SNAP / "metrics_4", SNAP / "metrics_maxsens"
for d in (METRICS4_DIR, MAXSENS_DIR):
    d.mkdir(parents=True, exist_ok=True)
for src in sorted((CFG_DIR / "metrics").glob("*.json")):
    cfg = json.loads(src.read_text())
    if cfg["metric_class"] == "max_sensitivity":
        cfg["params"]["nr_samples"] = MAXSENS_NR_SAMPLES
        (MAXSENS_DIR / src.name).write_text(json.dumps(cfg, indent=2))
    else:
        (METRICS4_DIR / src.name).write_text(json.dumps(cfg, indent=2))

# reference column -> (result column written by HM, direction: +1 higher is better, -1 lower is better)
METRICS = {
    "selectivity":              ("selectivity_pos",           -1),
    "attribution_localisation": ("attribution_localisation",  +1),
    "pointing_game":            ("pointing_game",             +1),
    "pf_auc5":                  ("pixel_flipping_pos_auc@5",  +1),
    "pf_auc20":                 ("pixel_flipping_pos_auc@20", +1),
    "max_sensitivity":          ("max_sensitivity",           -1),
}
cfgs = [json.loads(p.read_text()) for p in sorted([*METRICS4_DIR.glob("*.json"), *MAXSENS_DIR.glob("*.json")])]
pd.DataFrame([{"metric_name": c["name"], "quantus_class": c["metric_class"], "params": json.dumps(c["params"])}
              for c in cfgs]).to_csv(DATA_DIR / "metric_configs.csv", index=False)
pd.DataFrame([{"metric": k, "result_column": v[0], "direction": "↑" if v[1] > 0 else "↓"}
              for k, v in METRICS.items()]).to_csv(DATA_DIR / "metric_columns.csv", index=False)
print(f"thước đo: {len(cfgs)} config, {len(METRICS)} cột so với Bảng 1")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "metric_configs.csv", LABELS))
display(rep.show_csv(RUN_DIR, "metric_columns.csv", LABELS))

## 6. Sinh bản đồ giải thích

Cell đầu ghi `run_config.json` (mọi thứ worker cần biết) và định nghĩa `run_phase`. Hàm này mở `N_WORKERS` tiến trình `scripts/nb01_worker.py`, hiện thanh tiến trình tổng, chờ tất cả xong rồi ghi thời gian.

Trong mỗi worker, mỗi lần gọi HM chỉ xử lý một ảnh, nhờ tuỳ chọn chia ảnh có sẵn của HM: `split_id = k, num_splits = N_IMAGES` chọn đúng ảnh thứ k. Trước mỗi ảnh, seed được đặt thành 42·10⁴ + k. Bản đồ được lưu thành `seed42/<nhóm>/image_XXXX_<ảnh>/<phương pháp>.npz`; bản đồ nào đã có thì HM bỏ qua.

Thời gian được ghi theo hai kiểu:
- `seconds`: tổng thời gian tính toán của mọi worker;
- `wall_s`: thời gian đồng hồ của cả bước.

Một bước chỉ được ghi đè thời gian nếu lần chạy đó thật sự tính thêm được gì; nếu toàn cache thì giữ thời gian cũ.

HM bắt lỗi theo từng ảnh (ví dụ CUDA out of memory) rồi chạy tiếp mà không báo. Vì vậy cell kiểm tra dừng ngay nếu thiếu bản đồ nào, trước khi sang bước tính thước đo.

In [ ]:
RUN_CONFIG = {
    "n_images": N_IMAGES, "groups": GROUPS, "split_seed": SPLIT_SEED, "maxsens_seeds": MAXSENS_SEEDS,
    "innerloop_n": INNERLOOP_N, "hm_repo": str(HM_REPO), "imagenet": str(IMAGENET), "imagenet_s": str(IMAGENET_S),
    "main_dir": str(MAIN_DIR), "methods_dir": str(SNAP / "methods"),
    "metrics4_dir": str(METRICS4_DIR), "maxsens_dir": str(MAXSENS_DIR),
}
(RUN_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
WORKER_DIR = RUN_DIR / "workers"
WORKER_DIR.mkdir(exist_ok=True)

TIMING_CSV = DATA_DIR / "timing.csv"
timing = pd.read_csv(TIMING_CSV).to_dict("records") if TIMING_CSV.exists() else []

def record(rows):
    # replace the timing of a (phase, method) only when that pass computed something new
    global timing
    for r in rows:
        old = [t for t in timing if (t["phase"], t["method"]) == (r["phase"], r["method"])]
        if r["fresh"] or not old:
            timing = [t for t in timing if (t["phase"], t["method"]) != (r["phase"], r["method"])] + [r]
    pd.DataFrame(timing).to_csv(TIMING_CSV, index=False)

def run_phase(phase, label, units_per_image):
    # start the workers, draw one bar for all of them, wait, collect their timing
    n_w = min(N_WORKERS, N_IMAGES)
    for f in [*WORKER_DIR.glob(f"progress_{phase}_w*.json"), *WORKER_DIR.glob(f"timing_{phase}_w*.json")]:
        f.unlink()
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(GPU))
    procs, logs = [], []
    for w in range(n_w):
        logs.append(open(WORKER_DIR / f"{phase}_w{w}.log", "w", encoding="utf-8"))
        procs.append(subprocess.Popen([sys.executable, str(EXP_DIR / "scripts" / "nb01_worker.py"), str(RUN_DIR),
                                       phase, str(w), str(n_w)], stdout=logs[-1], stderr=subprocess.STDOUT,
                                      env=env, cwd=EXP_DIR))
        time.sleep(3)   # stagger start-up: model loading and HM's shared experiment_config.json
    t0 = time.perf_counter()
    with Progress(RUN_DIR, label, N_IMAGES * units_per_image, unit="lượt") as bar:
        while True:
            running = any(p.poll() is None for p in procs)
            done = 0
            for f in WORKER_DIR.glob(f"progress_{phase}_w*.json"):
                try:
                    done += json.loads(f.read_text(encoding="utf-8"))["done"]
                except (OSError, ValueError):
                    pass
            bar.update(done - bar.done)
            if not running:
                break
            time.sleep(2)
    wall = time.perf_counter() - t0
    for f in logs:
        f.close()
    failed = [w for w, p in enumerate(procs) if p.returncode != 0]
    if failed:
        tail = (WORKER_DIR / f"{phase}_w{failed[0]}.log").read_text(encoding="utf-8", errors="replace")[-3000:]
        raise RuntimeError(f"worker {failed} của bước {phase} lỗi; log: {WORKER_DIR}\n{tail}")
    rows = pd.DataFrame([r for f in sorted(WORKER_DIR.glob(f"timing_{phase}_w*.json"))
                         for r in json.loads(f.read_text(encoding="utf-8"))])
    rows = rows.groupby(["phase", "method"], as_index=False).agg(
        seconds=("seconds", "sum"), peak_gib=("peak_gib", "max"), fresh=("fresh", "any"))
    rows["per_image_s"], rows["wall_s"], rows["n_workers"] = rows["seconds"] / N_IMAGES, wall, n_w
    rows["cache"] = ~rows["fresh"]
    record(rows.to_dict("records"))
    for r in rows.itertuples():
        print(f"{r.phase:<16s} {r.method:<60s} {r.seconds:9.1f} s tính toán, đỉnh GPU {r.peak_gib:5.2f} GiB/worker"
              f"{'' if r.fresh else '  (cache)'}")
    print(f"{label}: {wall:.1f} s đồng hồ với {n_w} worker")

In [ ]:
run_phase("maps", "§6 sinh bản đồ", sum(len(files) for _, files in GROUPS.values()))

In [ ]:
def method_of(r, names):
    prefix = r.method_name if pd.isna(r.formula) else f"{r.method_name}_{r.formula}"
    hits = [n for n in names if n == prefix or n.startswith(prefix + "_")]
    assert len(hits) == 1, f"hàng {r.row}: tìm thấy {hits} cho tiền tố {prefix}"
    return hits[0]

def npz_path(r, img_dir_name, base=MAIN_DIR):
    return base / r.group / img_dir_name / f"{r.result_method}.npz"

REF["result_method"] = [method_of(r, sorted({p.stem for p in (MAIN_DIR / r.group).glob("image_*/*.npz")}))
                        for r in REF.itertuples()]
REF.to_csv(DATA_DIR / "row_to_method.csv", index=False)   # hàng Bảng 1 -> nhóm, phương pháp HM, số của bài
IMG_NAMES = sorted({d.name for g in GROUPS for d in (MAIN_DIR / g).glob("image_*")})
lacking = [(n, r.row) for n in IMG_NAMES for r in REF.itertuples() if not npz_path(r, n).exists()]
print(f"thư mục ảnh: {len(IMG_NAMES)} / {N_IMAGES}; bản đồ thiếu: {len(lacking)}")
assert len(IMG_NAMES) == N_IMAGES and not lacking, \
    f"thiếu bản đồ (ảnh, hàng), ví dụ {lacking[:3]}; xem log trong {WORKER_DIR} và evaluation.log trong {MAIN_DIR} " \
    f"(thường là CUDA out of memory: giảm N_WORKERS rồi chạy lại, bản đồ đã có được giữ)"

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "row_to_method.csv", LABELS)[[LABELS["columns"][c] for c in
        ["row", "label", "layer", "group", "result_method"]]])
t = rep.show_csv(RUN_DIR, "timing.csv", LABELS)
display(t[t[LABELS["columns"]["phase"]] == "maps"].round(2))

## 7. Tính 4 thước đo (Selectivity, Attr. Loc., Pointing Game, Pixel Flipping)

HM chấm các thước đo cho mọi bản đồ và ghi kết quả vào `results.csv` của từng ảnh. Bốn thước đo này tất định: không dùng số ngẫu nhiên. Việc kiểm tra đủ ô nằm ở §9, sau khi Max Sens cũng đã chạy xong.

In [ ]:
run_phase("metrics4", "§7 4 thước đo", len(GROUPS))

## 8. Max Sensitivity

Max Sens tính lại bản đồ trên 25 ảnh có nhiễu (chạy thử: 2), nên đây là bước tốn thời gian nhất. Nhiễu lấy từ `np.random`, với seed của ảnh k là 42·10⁴ + k. Như HM, ta chỉ chạy seed 42. Nếu muốn đo thêm dao động Monte Carlo, thêm seed vào `MAXSENS_SEEDS`: mỗi seed bổ sung có thư mục `maxsens_seed{s}/` riêng, trỏ tới các bản đồ của §6 bằng hard link.

In [ ]:
run_phase("maxsens", "§8 Max Sens", len(MAXSENS_SEEDS) * len(GROUPS))

## 9. Thu thập kết quả và kiểm tra đủ ô

Gom `results.csv` của mọi ảnh vào `data/per_image_seed42.csv`, rồi chuyển sang dạng dài (ảnh × hàng × thước đo) trong `data/long_seed42.csv`.

HM **nuốt lỗi** khi tính thước đo: nó chỉ ghi lỗi vào log rồi để trống kết quả. Vì vậy cell kiểm tra mọi ô **có số trong Bảng 1** đều có giá trị, và dừng lại nếu thiếu ô nào.

In [ ]:
def collect(base):
    return pd.concat([ResultsManager.collect_all_results(base / g).assign(group=g) for g in GROUPS
                      if (base / g).exists()], ignore_index=True)

def long_table(df):
    # one row per (image, Table-1 row, metric)
    out = []
    for r in REF.itertuples():
        sub = df[(df["group"] == r.group) & (df["method"] == r.result_method)]
        for key, (col, _) in METRICS.items():
            if col in sub:
                out.append(pd.DataFrame({"image_name": sub["image_name"].values, "row": r.row, "metric": key,
                                         "value": pd.to_numeric(sub[col], errors="coerce").values}))
    return pd.concat(out, ignore_index=True)

PER_IMAGE = collect(MAIN_DIR)
PER_IMAGE.to_csv(DATA_DIR / f"per_image_seed{SPLIT_SEED}.csv", index=False)
LONG = long_table(PER_IMAGE)
LONG.to_csv(DATA_DIR / f"long_seed{SPLIT_SEED}.csv", index=False)   # (ảnh, hàng, thước đo) -> giá trị

# every (row, metric) that has a number in Table 1 must have a value for every image
REQUIRED = REF.melt(id_vars=["row"], value_vars=list(METRICS), var_name="metric").dropna()[["row", "metric"]]
images = sorted(PER_IMAGE["image_name"].unique())
expected = pd.MultiIndex.from_tuples([(i, r, m) for i in images for r, m in REQUIRED.itertuples(index=False)],
                                     names=["image_name", "row", "metric"])
have = LONG.dropna(subset=["value"]).set_index(["image_name", "row", "metric"]).index
missing = expected.difference(have)
print(f"ảnh: {len(images)} / {N_IMAGES}, ô có giá trị: {len(expected) - len(missing)} / {len(expected)}")
assert len(images) == N_IMAGES, f"thiếu ảnh; xem evaluation.log trong {MAIN_DIR}"
assert len(missing) == 0, f"thiếu {len(missing)} ô, ví dụ {list(missing[:5])}; xem evaluation.log trong {MAIN_DIR}"

# Max Sens per seed (seed 42 from the main run, extra seeds from their own folders)
need = REQUIRED[REQUIRED.metric == "max_sensitivity"]["row"]
MS = [LONG[LONG.metric == "max_sensitivity"].assign(seed=SPLIT_SEED)]
for seed in [s for s in MAXSENS_SEEDS if s != SPLIT_SEED]:
    df = collect(RUN_DIR / f"maxsens_seed{seed}")
    df.to_csv(DATA_DIR / f"per_image_maxsens_seed{seed}.csv", index=False)
    ms = long_table(df)
    ms = ms[ms.metric == "max_sensitivity"].assign(seed=seed)
    assert ms[ms.row.isin(need)]["value"].notna().sum() == N_IMAGES * len(need), f"seed {seed}: thiếu ô Max Sens"
    MS.append(ms)
MS = pd.concat(MS, ignore_index=True)
MS.to_csv(DATA_DIR / "maxsens_long.csv", index=False)
MS[MS.row.isin(need)].groupby(["row", "seed"])["value"].mean().unstack("seed").to_csv(DATA_DIR / "maxsens_seed_means.csv")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
n = pd.read_csv(DATA_DIR / f"long_seed{SPLIT_SEED}.csv").groupby(["row", "metric"])["value"].count()
n = n.rename("n_values").reset_index().pivot(index="row", columns="metric", values="n_values")[rep.METRIC_KEYS]
n.columns = [LABELS["metrics"][c] for c in n.columns]
n.index.name = LABELS["columns"]["row"]
print("số ảnh có giá trị cho mỗi ô"); display(n)
print(LABELS["seed_table"]); display(rep.maxsens_seed_table(RUN_DIR, LABELS).round(4))

## 10. Tổng hợp theo dạng Bảng 1

Với mỗi ô ta tính trung bình theo ảnh, độ lệch chuẩn $s$, số ảnh $n$ và $\mathrm{SE} = s/\sqrt n$. Nếu `MAXSENS_SEEDS` có nhiều hơn một seed, $s_\text{seed}$ (độ lệch chuẩn của trung bình giữa các seed) được cộng vào SE của Max Sens. Với một seed thì $s_\text{seed} = 0$. Tất cả lưu trong `data/summary_stats.csv`.

In [ ]:
stats = LONG.groupby(["row", "metric"])["value"].agg(mean="mean", sd="std", n="count").reset_index()
stats["se"] = stats["sd"] / np.sqrt(stats["n"])
seed_means = MS.groupby(["row", "seed"])["value"].mean().unstack("seed")
s_seed = (seed_means.std(axis=1, ddof=1) if seed_means.shape[1] > 1 else seed_means.iloc[:, 0] * 0).rename("s_seed")
stats = stats.merge(s_seed, left_on="row", right_index=True, how="left")
stats.loc[stats.metric != "max_sensitivity", "s_seed"] = 0.0
stats["s_seed"] = stats["s_seed"].fillna(0.0)
stats["se_eff"] = np.sqrt(stats["se"] ** 2 + stats["s_seed"] ** 2)

paper = REF.melt(id_vars=["row"], value_vars=list(METRICS), var_name="metric", value_name="paper")
COMP = stats.merge(paper, on=["row", "metric"]).merge(REF[["row", "label", "layer"]], on="row")
COMP["delta"] = COMP["mean"] - COMP["paper"]
COMP.to_csv(DATA_DIR / "summary_stats.csv", index=False)   # mean, sd, n, se, s_seed, se_eff, paper, delta
print(f"tổng hợp: {len(COMP)} ô")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
for v in ("mean", "paper", "delta"):
    print(LABELS["table_values"][v]); display(rep.table1(RUN_DIR, v, LABELS).round(3))

## 11. Đánh giá tự động

- **Mức 1:** từng ô so với ngưỡng 2·SE.
- **Mức 2 (a), hàng dẫn đầu:** tìm hàng HM dẫn đầu và hàng ta dẫn đầu trong các hàng 1–8. Nếu hai hàng khác nhau, xét hiệu ghép cặp theo ảnh giữa chúng; hiệu nằm trong 2·SE thì tính là hoà.
- **Mức 2 (b), Split so với LayerCAM:** lấy hàng Split tốt nhất và hàng LayerCAM tốt nhất theo Bảng 1. Kết luận chỉ bị coi là đảo khi hiệu ghép cặp của ta ngược dấu với bài **và** vượt 2·SE.
- **Mức 2 (c):** Random phải có trung bình thấp nhất ở Attr. Loc. và Point.

Thước đo không có số trong bài (Max Sens của Random) không được chấm. Kết quả lưu ở `data/level1.csv`, `data/level2.csv`, `nb01_verdict.json` và `nb01_verdict.md`.

In [ ]:
SPLIT_ROWS, LAYERCAM_ROWS, RANDOM_ROW = [1, 2, 3, 4, 5, 6], [7, 8], 9
WIDE = LONG.pivot_table(index=["metric", "image_name"], columns="row", values="value")

# Level 1
L1 = COMP.dropna(subset=["paper"]).copy()
L1["tol"] = 2 * L1["se_eff"]
L1["z"] = (L1["delta"].abs() / L1["se_eff"]).replace([np.inf], np.nan)
L1["passed"] = L1["delta"].abs() <= L1["tol"] + 1e-12
L1["note"] = np.where(L1["passed"] & (L1["z"] > 0.5), "đạt, cần ghi chú (z > 0.5)",
                      np.where(L1["passed"], "", "TRƯỢT"))

def paired(metric, a, b):
    d = (WIDE.loc[metric][a] - WIDE.loc[metric][b]).dropna()
    return d.mean(), (d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan)

def best(values, rows, sign):
    v = values.loc[rows]
    return int(v.idxmax() if sign > 0 else v.idxmin())

# Level 2
L2 = []
for key, (_, sign) in METRICS.items():
    ours = COMP[COMP.metric == key].set_index("row")["mean"]
    theirs = COMP[COMP.metric == key].set_index("row")["paper"]
    # (a) leader among rows 1-8
    hp, hm = best(theirs, SPLIT_ROWS + LAYERCAM_ROWS, sign), best(ours, SPLIT_ROWS + LAYERCAM_ROWS, sign)
    if hp == hm:
        ok, note = True, f"cùng dẫn đầu: hàng {hp}"
    else:
        diff, se = paired(key, hm, hp)
        ok = bool(abs(diff) <= 2 * se)
        note = f"HM: hàng {hp}, ta: hàng {hm}; hiệu ghép cặp {diff:+.4f} (2·SE = {2 * se:.4f}) → {'hoà' if ok else 'khác'}"
    L2.append({"metric": key, "check": "(a) dẫn đầu", "passed": ok, "detail": note})
    # (b) best Split vs best LayerCAM, both chosen by the paper's numbers
    s, l = best(theirs, SPLIT_ROWS, sign), best(theirs, LAYERCAM_ROWS, sign)
    paper_sign = np.sign(sign * (theirs[s] - theirs[l]))
    diff, se = paired(key, s, l)
    reversed_ = bool(np.sign(sign * diff) == -paper_sign and abs(diff) > 2 * se)
    note = (f"hàng {s} − hàng {l}: HM {'Split tốt hơn' if paper_sign > 0 else 'LayerCAM tốt hơn'}; "
            f"ta {diff:+.4f} ± {2 * se:.4f} (2·SE)")
    L2.append({"metric": key, "check": "(b) Split vs LayerCAM", "passed": not reversed_, "detail": note})
for key in ["attribution_localisation", "pointing_game"]:
    ours = COMP[COMP.metric == key].set_index("row")["mean"]
    ok = bool(ours[RANDOM_ROW] < ours.drop(RANDOM_ROW).min())
    L2.append({"metric": key, "check": "(c) Random kém nhất", "passed": ok,
               "detail": f"Random {ours[RANDOM_ROW]:.3f}, thấp nhất còn lại {ours.drop(RANDOM_ROW).min():.3f}"})
L2 = pd.DataFrame(L2)

l1_ok, l2_ok = bool(L1["passed"].all()), bool(L2["passed"].all())
verdict = "THÀNH CÔNG" if (l1_ok and l2_ok) else ("MỘT PHẦN" if l2_ok else "THẤT BẠI")
if N_IMAGES < 2:
    verdict = "KHÔNG CHẤM ĐƯỢC (1 ảnh: không tính được SE)"
label = verdict + (" (CHẠY THỬ, không có giá trị kết luận)" if SMOKE else "")

cols = ["row", "label", "layer", "metric", "paper", "mean", "delta", "se", "s_seed", "tol", "z", "passed", "note"]
L1[cols].to_csv(DATA_DIR / "level1.csv", index=False)
L2.to_csv(DATA_DIR / "level2.csv", index=False)
summary = {"verdict": verdict, "label": label, "smoke": SMOKE, "n_images": N_IMAGES, "dtype": "float32",
           "level1_pass": int(L1["passed"].sum()), "level1_total": len(L1),
           "level1_flagged": int((L1["note"] == "đạt, cần ghi chú (z > 0.5)").sum()),
           "level2_pass": int(L2["passed"].sum()), "level2_total": len(L2),
           "maxsens_nr_samples": MAXSENS_NR_SAMPLES, "maxsens_seeds": MAXSENS_SEEDS, "env": ENV}
(RUN_DIR / f"{NB_ID}_verdict.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

report = [f"# Kết luận tái lập Bảng 1 (VGG16): **{label}**", "",
          f"- Số ảnh: {N_IMAGES} / {N_TEST}; dtype float32 (bài ghi float64); Max Sens: {MAXSENS_NR_SAMPLES} mẫu, seed {MAXSENS_SEEDS}",
          f"- Mức 1: {summary['level1_pass']}/{summary['level1_total']} ô đạt, {summary['level1_flagged']} ô cần ghi chú",
          f"- Mức 2: {summary['level2_pass']}/{summary['level2_total']} kiểm tra đạt", "",
          "## Mức 1", "", rep.show_csv(RUN_DIR, "level1.csv").round(4).to_markdown(index=False), "",
          "## Mức 2", "", rep.show_csv(RUN_DIR, "level2.csv").to_markdown(index=False)]
(RUN_DIR / f"{NB_ID}_verdict.md").write_text("\n".join(report), encoding="utf-8")
print(f"kết luận: {label}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
v = json.loads((RUN_DIR / f"{NB_ID}_verdict.json").read_text(encoding="utf-8"))
display(Markdown(f"### Kết luận: **{v['label']}**\n\nMức 1: {v['level1_pass']}/{v['level1_total']} ô đạt "
                 f"({v['level1_flagged']} ô cần ghi chú) · Mức 2: {v['level2_pass']}/{v['level2_total']} kiểm tra đạt"))
display(rep.show_csv(RUN_DIR, "level1.csv", LABELS).round(4))
display(rep.show_csv(RUN_DIR, "level2.csv", LABELS))

## 12. Kiểm tra trực quan

Cell tính toán kiểm tra mọi bản đồ: không chứa NaN, không phải hằng số. Nó cũng lưu số liệu của lưới hình (`data/fig_maps_grid.npz`): bản đồ của tối đa 3 ảnh đầu, lấy trung bình theo kênh và chia cho $\max|\cdot|$. Cell hiển thị vẽ lưới từ file đó; mỗi hàng là một ảnh, mỗi cột là một phương pháp.

In [ ]:
checks = []
for n in IMG_NAMES:
    for r in REF.itertuples():
        e = np.load(npz_path(r, n))["explanation"]
        checks.append({"image": n, "row": r.row, "finite": bool(np.isfinite(e).all()), "std": float(np.nanstd(e))})
MAP_CHECKS = pd.DataFrame(checks)
MAP_CHECKS["passed"] = MAP_CHECKS["finite"] & (MAP_CHECKS["std"] > 0)
MAP_CHECKS.to_csv(DATA_DIR / "map_checks.csv", index=False)

show_names = IMG_NAMES[:min(3, len(IMG_NAMES))]
maps, originals = [], []
for n in show_names:
    o = plt.imread(MAIN_DIR / REF.group.iloc[0] / n / "original_image.png")[..., :3]
    originals.append(o if o.dtype == np.uint8 else (o * 255).round().astype(np.uint8))
    row_maps = []
    for r in REF.itertuples():
        e = np.load(npz_path(r, n))["explanation"]
        e = e.mean(0) if e.ndim == 3 else e
        row_maps.append((e / (np.abs(e).max() + 1e-12)).astype(np.float32))
    maps.append(row_maps)
np.savez_compressed(DATA_DIR / "fig_maps_grid.npz", maps=np.array(maps), originals=np.stack(originals),
                    rows=REF["row"].to_numpy(), image_names=np.array(show_names))
print(f"bản đồ lỗi (NaN hoặc hằng): {int((~MAP_CHECKS['passed']).sum())} / {len(MAP_CHECKS)}")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
mc = rep.show_csv(RUN_DIR, "map_checks.csv", LABELS)
bad = ~mc[LABELS["columns"]["passed"]].astype(bool)
display(mc[bad] if bad.any() else Markdown("Mọi bản đồ hợp lệ."))
rep.plot_maps_grid(RUN_DIR, LABELS); plt.show()

## 13. Thời gian chạy và ước lượng cho 566 ảnh

Thời gian đo được ngoại suy lên 566 ảnh với 25 mẫu Max Sens. Lần chạy thử dùng `MAXSENS_NR_SAMPLES` mẫu, nên phần Max Sens được nhân thêm hệ số 25/`MAXSENS_NR_SAMPLES`. Có hai ước lượng:
- **tính toán**: tổng thời gian của mọi worker, tức là thời gian nếu chỉ có 1 tiến trình;
- **đồng hồ**: thời gian thật với `N_WORKERS` worker như lần chạy này.

Lần chạy thử với ít ảnh có chi phí khởi động (nạp model, tách mạng) chiếm tỉ lệ lớn, nên ước lượng đồng hồ thường bi quan.

In [ ]:
T = pd.DataFrame(timing)
by_phase = T.assign(group=T["phase"].where(~T["phase"].str.startswith("maxsens"), "maxsens")) \
            .groupby("group").agg(compute=("seconds", "sum"), wall=("wall_s", "first"))
scale = {"maps": 1.0, "metrics4": 1.0, "maxsens": 25 / MAXSENS_NR_SAMPLES}
compute_h = sum(by_phase.loc[p, "compute"] * scale[p] for p in by_phase.index) / N_IMAGES * N_TEST / 3600
wall_h = sum(by_phase.loc[p, "wall"] * scale[p] for p in by_phase.index) / N_IMAGES * N_TEST / 3600
n_w = int(T["n_workers"].max())
pd.DataFrame([
    {"quantity": "sinh bản đồ (s/ảnh, tính toán)", "value": by_phase.loc["maps", "compute"] / N_IMAGES},
    {"quantity": "4 thước đo (s/ảnh, tính toán)", "value": by_phase.loc["metrics4", "compute"] / N_IMAGES},
    {"quantity": f"Max Sens, {MAXSENS_NR_SAMPLES} mẫu (s/ảnh, tính toán)", "value": by_phase.loc["maxsens", "compute"] / N_IMAGES},
    {"quantity": "đỉnh bộ nhớ GPU mỗi worker (GiB)", "value": float(T["peak_gib"].max())},
    {"quantity": "ước lượng chạy đủ 566 ảnh, 1 tiến trình (giờ)", "value": compute_h},
    {"quantity": f"ước lượng chạy đủ 566 ảnh, {n_w} worker, đồng hồ (giờ)", "value": wall_h},
    ]).assign(gpu=ENV["gpu"]).to_csv(DATA_DIR / "runtime_estimate.csv", index=False)
print(f"ước lượng chạy đủ trên {ENV['gpu']}: {compute_h:.1f} giờ với 1 tiến trình, {wall_h:.1f} giờ đồng hồ với {n_w} worker")

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
display(rep.show_csv(RUN_DIR, "timing.csv", LABELS).round(2))
display(rep.show_csv(RUN_DIR, "runtime_estimate.csv", LABELS).round(2))

## 14. Đổi nhãn và trình bày lại

Mọi cell hiển thị (dòng đầu `# ▶ Hiển thị`) chỉ đọc file trong `data/` và dùng `LABELS` cho tên cột, tên hàng và tiêu đề. Để đổi nhãn mà không tính lại:
1. Nếu kernel vừa khởi động lại, chạy §1 (chỉ đặt đường dẫn, không tính gì).
2. Sửa `LABELS` ở cell dưới và chạy nó.
3. Chạy lại cell hiển thị cần thiết, hoặc cell cuối để vẽ lại tất cả. Hình được ghi đè vào `figures/`, bảng vào `tables/`.

Danh sách các khoá sửa được: `rep.default_labels()`.

| File trong `data/` | Nội dung |
| --- | --- |
| `params.csv`, `env.json` | tham số và môi trường (GPU, bộ nhớ trống lúc bắt đầu, số worker) |
| `data_checks.csv`, `image_split.csv`, `fig_example_mask.npz` | kiểm tra dữ liệu, danh sách 50 val + 566 test, hình ví dụ |
| `method_table.csv`, `row_to_method.csv` | hàng Bảng 1 ↔ config ↔ tên phương pháp trong kết quả HM, số của bài |
| `metric_configs.csv`, `metric_columns.csv` | tham số thước đo, cột kết quả, hướng |
| `per_image_seed42.csv`, `long_seed42.csv` | điểm từng ảnh (dạng HM và dạng dài) |
| `maxsens_long.csv`, `maxsens_seed_means.csv` | Max Sens theo seed |
| `summary_stats.csv` | trung bình, SE, số của bài, Δ cho mọi ô |
| `level1.csv`, `level2.csv` | đánh giá mức 1 và mức 2 (kết luận: `nb01_verdict.json`, `nb01_verdict.md` ở thư mục chạy) |
| `map_checks.csv`, `fig_maps_grid.npz` | kiểm tra bản đồ, số liệu lưới hình |
| `timing.csv`, `runtime_estimate.csv` | thời gian (tính toán, đồng hồ, đỉnh bộ nhớ GPU mỗi worker) |

In [ ]:
import importlib
importlib.reload(rep)
LABELS = rep.default_labels()
# ví dụ đổi nhãn:
# LABELS["metrics"]["pf_auc5"] = "Pixel Flipping AUC@5 ↑"
# LABELS["rows"][7] = "LayerCAM (tầng 14)"
# LABELS["columns"]["passed"] = "pass"
# LABELS["fig_maps"]["title"] = "SplitCAM, SplitGrad, SplitLRP và LayerCAM trên VGG16"

In [ ]:
# ▶ Hiển thị: chỉ đọc từ data/
for v in ("mean", "paper", "delta"):
    print(LABELS["table_values"][v]); display(rep.table1(RUN_DIR, v, LABELS).round(3))
print(LABELS["seed_table"]); display(rep.maxsens_seed_table(RUN_DIR, LABELS).round(4))
display(Markdown(f"### Kết luận: **{json.loads((RUN_DIR / f'{NB_ID}_verdict.json').read_text(encoding='utf-8'))['label']}**"))
display(rep.show_csv(RUN_DIR, "level2.csv", LABELS))
rep.plot_example_mask(RUN_DIR, LABELS); plt.show()
rep.plot_maps_grid(RUN_DIR, LABELS); plt.show()